
<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img
    src="https://databricks.com/wp-content/uploads/2018/03/db-academy-rgb-1200px.png"
    alt="Databricks Learning"
  >
</div>


# 6 - Monitoring and Repairing Tasks

In this lesson, you will learn how to monitor jobs, intentionally cause a failure, and repair failed runs in Databricks.

**This demo covers:**
- How to repair a failed run
- Rerunning only failed tasks
- Adding a dashboard task

## Learning Objectives

By the end of this lesson, you should be able to:
- Monitor failed job runs
- Repair and re-run failed tasks

![Lesson06_full_run](./Includes/images/Lesson06_full_run.png)

After completing this demo, your job will look like above.

## REQUIRED - SELECT CLASSIC COMPUTE (The cluster named 'labuser')

Before executing cells in this notebook, please select your classic compute cluster in the lab. Be aware that **Serverless** is enabled by default.

Follow these steps to select the classic compute cluster:


1. Navigate to the top-right of this notebook and click the drop-down menu to select your cluster. By default, the notebook will use **Serverless**.

2. If your cluster is available, select it and continue to the next cell. If the cluster is not shown:

   - Click **More** in the drop-down.

   - In the **Attach to an existing compute resource** window, use the first drop-down to select your unique cluster.

**NOTE:** If your cluster has terminated, you might need to restart it in order to select it. To do this:

1. Right-click on **Compute** in the left navigation pane and select *Open in new tab*.

2. Find the triangle icon to the right of your compute cluster name and click it.

3. Wait a few minutes for the cluster to start.

4. Once the cluster is running, complete the steps above to select your cluster.

## A. Classroom Setup

Run the following cell to configure your working environment for this course. It will also set your default catalog to **dbacademy** and the schema to your specific schema name shown below using the `USE` statements.
<br></br>
```
USE CATALOG dbacademy;
USE SCHEMA dbacademy.<your unique schema name>;
```

**NOTE:** The **DA** object is only used in Databricks Academy courses and is not available outside of these courses.

In [0]:
%run ./Includes/Classroom-Setup-6

/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


/databricks/python_shell/lib/lsp_backend/line_magic_sanitizer.py:98: UserWarning: `make_tokens_by_line` received a list of lines which do not have lineending markers ('\n', '\r', '\r\n', '\x0b', '\x0c'), behavior will be unspecified
  tokens = make_tokens_by_line(lines)


Course Catalog:,
Your Schema:,


Data has been copied from /Volumes/dbacademy_retail/v01/source_files/customers.csv to /Volumes/dbacademy/labuser15105141_1778791941/trigger_storage_location/


## B. Explore Your Schema
Complete the following to explore your **dbacademy.labuser** schema:

1. In the left navigation bar, select the catalog icon:  ![Catalog Icon](./Includes/images/catalog_icon.png)

2. Locate the catalog called **dbacademy** and expand the catalog.

3. Expand your **labuser** schema. 

4. Notice that within your schema you will find the tables created by our job till now.


## C. View Your Files

You can find the notebook in **Task Files** > **Lesson 6 Files**. Use the link below to view and explore the code:
[Task Files/Lesson 6 Files/6.1 - Transforming Customers Orders State Wise Data]($./Task Files/Lesson 6 Files/6.1 - Transforming Customers Orders State Wise Data)

- We have ingested three tables into our job. Next, we will add tasks (as notebooks) that will clean and transform the tables created by the For Each loop. These notebooks demonstrate how we can apply transformation logic specific to each table.

At the end, we will also add a dashboard, which you can find under Lesson 6 Task Files: [Task Files/Lesson 6 Files]($./Task Files/Lesson 6 Files)

## D. Creating the Starter Job

Next, we will add the above task Notebook to our job. We'll programmatically creating  using the SDK. Simply run the next command to create and configure your job, even if you haven't completed any previous demos. These commands will set up your job with all work completed so far.

In [0]:
DA.Demo_6_starter_job()

Created the Job Demo_06_Retail_Job_labuser15105141_1778791941


## E. Adding a Notebook Task to our Retail Job 
1. Right-click the **Jobs and Pipelines** button in the sidebar and select *Open Link in New Tab*.

2. Locate your job named **Demo_06_Retail_Job_<-your schema name->**.

3. Navigate to the **Tasks** tab and click **Add task**, then select **Notebook**. Configure the task with the following settings:

| Setting      | Instructions |
|--------------|-------------|
| **Task name**    | Enter **transforming_customers_orders_data** |
| **Type**         | Ensure **Notebook** is selected |
| **Source**       | Ensure **Workspace** is selected |
| **Path**         | Use the navigator to select [./task files/Lesson 6/6.1 - Transforming Customers Orders State Wise Data]($./Task Files/Lesson 6 Files/6.1 - Transforming Customers Orders State Wise Data) under **Lesson 6 Files** |
| **Compute**      | Select **Serverless** |
| **Depends on**   | Choose **customers_orders_state_wise_report_iterator** |
| **Dependencies** | Set to **All Succeeded** |
| **Retries** | Click on Retries and **untick** "Enable serverless auto-optimization (may include at most 3 retries)" **to disable retries**.

4. Click on **Create Task**.

![Lesson06_notebook_task.png](./Includes/images/Lesson06_notebook_task.png)


##F. Running your Job
1. Run the job by clicking **Run now** in the upper-right corner. 

2. A pop-up window appears with a link to the job run. Click **View run**.
    - **NOTE:** You can also select the **Runs** tab and then select the link under **Start time** to view the job run **DAG**.

3. Watch the tasks in the DAG. The colors change to show the progress of the task (about 2-3 minutes to complete):

    * **Gray** -- the task has not started
    * **Green stripes** -- the task is currently running
    * **Solid green** -- the task completed successfully
    * **Dark red** -- the task failed
    * **Light red** -- an upstream task failed, so the current task never ran

4. When the run is finished, note that **transforming_customers_orders_data** failed. This was expected.

![Lesson06_fail_run.png](./Includes/images/Lesson06_fail_run.png)


## G. Reparando Execuções de Jobs

Você pode visualizar os notebooks usados em uma tarefa, incluindo sua saída, como parte de uma execução de job. Isso ajuda a diagnosticar erros. Também é possível reexecutar tarefas específicas em uma execução de job que falhou.

Considere este exemplo:

Você está desenvolvendo um job com vários notebooks. Durante uma execução, uma das tarefas falha. Você pode atualizar o código nesse notebook e reexecutar a tarefa que falhou e quaisquer tarefas que dependem dela. Também é possível alterar parâmetros da tarefa e reexecutá-la. Veja como fazer isso:

1. No canto superior direito, clique em **Repair run**.

2. Abra [Task Files/Lesson 6 Files/6.1 - Transforming Customers Orders State Wise Data]($./Task Files/Lesson 6 Files/6.1 - Transforming Customers Orders State Wise Data) e observe que a função `def clean_common` usa o nome de coluna incorreto.

3. Atualize o nome da coluna de `customer` para `customer_name` no script de código.

![Lesson06_script_snip.png](./Includes/images/Lesson06_script_snip.png)

4. Volte para a execução do seu job. No canto superior direito, clique em **Repair run**. Certifique-se de que a tarefa **transforming_customers_orders_data** está selecionada.

5. O "1" no botão **Repair run** indica que o Databricks selecionou tanto a tarefa que falhou quanto a tarefa dependente. Você pode selecionar ou desmarcar quaisquer tarefas que deseja reexecutar.

6. Aguarde a conclusão da execução.

## H. Reviewing the Run

Once your job run is successful, go back to your job and click on the **Runs** tab. Click on the latest run, then click on **transforming_customers_orders_data**. In the top left corner, you will notice the run status, which is a dropdown list. By clicking on each dropdown option, you can see the output of your task run. Notice the code difference between the successful and failed tasks. You should see the correct column name (`customer_name`) in the successful run.

![Lesson06_review_run.png](./Includes/images/Lesson06_review_run.png)

## I. Adding a Dashboard to Your Job

In this section, we will integrate a pre-created dashboard into your job. The dashboard has been prepared for you and is stored as a JSON input file, so you don't need to create it from scratch.

#### I1. Configuring Your Retail Dashboard
Follow these steps to view and configure your dashboard:

1. Navigate to **Lesson 6 Files**: [Task Files/Lesson 6 Files]($./Task Files/Lesson 6 Files) to locate the input file.
2. Locate the **input_file**. This file will assist in generating a user-specific dashboard JSON file.
3. Run the command below to automatically create the dashboard from the input file.

In [0]:
DA.dashboard_creation_from_input()

Updated dashboard exported to /Workspace/Users/labuser15105141_1778791941@vocareum.com/deploy-workloads-with-lakeflow-jobs-en_us-3.2.4/Deploy Workloads with Lakeflow Jobs/Task Files/Lesson 6 Files/labuser15105141_1778791941_Retail_Dashboard.json
Retail Dashboard have been created successfully!

Your Dashboard name is labuser15105141_1778791941_Retail_Dashboard


4. Clique no ícone de pasta ![folder_icon.png](./Includes/images/folder_icon.png) à esquerda; você verá seu dashboard junto com outros notebooks de laboratório/demonstração.
5. Certifique-se de que o nome do dashboard corresponde ao mostrado na saída da célula acima. Clique no dashboard para abri-lo.
6. No canto superior esquerdo, clique em **Editar rascunho** para visualizar e modificar seu dashboard.
7. No canto superior direito, garanta que **shared_warehouse** esteja selecionado como recurso de computação. Não use "unknown warehouse". Aguarde o SQL Warehouse iniciar para que seu dashboard possa ser renderizado.
8. Clique na guia **Dados** à esquerda. Verifique se as seguintes tabelas estão selecionadas:
   - `customers_orders_ny_gold`
   - `customers_orders_va_gold`
   - `customers_sales_summary_gold`
9. Execute consultas em cada tabela na caixa de consulta para obter insights. Certifique-se de que seu catálogo está definido como **dbacademy** e seu schema é seu **labuser** específico.
10. Clique em **Publicar**. Aqui, você encontrará opções para compartilhar permissões do dashboard. Seu dashboard foi publicado automaticamente pelo comando acima. Mas, se você fizer alterações, certifique-se de publicar o dashboard novamente para atualizá-lo.

      **Nota:** Publicar o dashboard é essencial para usá-lo nas tarefas do seu job.

Isso garante que seu dashboard esteja conectado aos dados e recursos de computação corretos.

#### I2. Adding a Dashboard Task to Your Job
This dashboard was precreated for you within the classroom setup script. Creating dashboards are outside the scope of the course.

1. In your job, click **Add task** and select **Dashboard**. Configure the task as follows:

| Setting        | Instructions                                                                 |
|----------------|------------------------------------------------------------------------------|
| Task name      | Enter **refreshing_retail_dashboard**                                        |
| Type           | Ensure **Dashboard** is selected                                             |
| Dashboard      | Select your dashboard from the list of available dashboards. The name of your dashboard will be shown in the output cell of the dashboard creation command.                                                |
| SQL warehouse  | Select your **Warehouse** from the dropdown                                  |
| Subscribers    | Select email address from drop-down to receive dashboard snapshots. In our learning environment, you will not be able to add additional emails.                    |
| Depends on     | Select **transforming_customers_sales_table** and **transforming_customers_orders_data** |
| Dependencies   | Set to **All Succeeded**                                                     |

2. Click on **Save task**.

![Lesson06_dashboard_task.png](./Includes/images/Lesson06_dashboard_task.png)

3. Click on **Run Now** and wait for run to get completed.


**NOTE:** Please ensure that your dashboard's data section includes all gold tables (tables with the `gold` suffix from your schema) and that your dashboard is connected to your SQL Warehouse.

#### I3. Analyze Retail Dashboard
After the run is complete, check your email. You should have received an email from Databricks containing the Retail_Dashboard.

## J. Querying Lakehouse System Tables
Databricks provides system catalogs that contain metadata about billing, access, Lakehouse operations, compute resources, and more. For this course, we will focus on querying the billing and Lakehouse operation details.

1. Start by seeing which different schemas are present under the catalog `system`.

In [0]:
%sql
SHOW SCHEMAS IN system

databaseName
access
ai
compute
data_quality_monitoring
information_schema
lakeflow
query


2. Begin by viewing the different tables available in the `lakeflow` schema.

In [0]:
%sql
SHOW TABLES IN system.lakeflow

database,tableName,isTemporary
lakeflow,job_run_timeline,false
lakeflow,job_task_run_timeline,false
lakeflow,job_tasks,false
lakeflow,jobs,false
lakeflow,pipeline_update_timeline,false
lakeflow,pipelines,false
lakeflow,zerobus_ingest,false
lakeflow,zerobus_stream,false
,_sqldf,true


3. Let's join the `jobs` and `job_task_run_timeline` tables to gain insights about recently executed jobs.

In [0]:
%sql
SELECT jobs.workspace_id, 
        jobs.name as job_name,
        jobs.job_id,
        timeline.run_id,
        timeline.period_start_time,
        timeline.period_end_time,
        timeline.task_key,
        timeline.result_state 
FROM system.lakeflow.jobs as jobs
INNER JOIN
system.lakeflow.job_task_run_timeline as timeline
ON jobs.job_id = timeline.job_id
WHERE lower(jobs.name) LIKE 'demo_06_retail_job_%'
ORDER BY timeline.period_start_time


workspace_id,job_name,job_id,run_id,period_start_time,period_end_time,task_key,result_state
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,558642850860529,2025-08-26T16:59:39.464Z,2025-08-26T17:01:37.691Z,ingesting_sales,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,558642850860529,2025-08-26T16:59:39.464Z,2025-08-26T17:01:37.691Z,ingesting_sales,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,946779211470915,2025-08-26T16:59:39.464Z,2025-08-26T17:01:06.496Z,ingesting_customers,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,946779211470915,2025-08-26T16:59:39.464Z,2025-08-26T17:01:06.496Z,ingesting_customers,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,642468218927216,2025-08-26T16:59:39.464Z,2025-08-26T17:01:30.977Z,ingesting_orders,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,642468218927216,2025-08-26T16:59:39.464Z,2025-08-26T17:01:30.977Z,ingesting_orders,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,553017387601635,2025-08-26T17:01:31.275Z,2025-08-26T17:02:27.525Z,customers_orders_report,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,553017387601635,2025-08-26T17:01:31.275Z,2025-08-26T17:02:27.525Z,customers_orders_report,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,651079287181128,2025-08-26T17:01:37.978Z,2025-08-26T17:02:37.681Z,customers_sales_summary,SUCCEEDED
1116232396022739,Demo_06_Retail_Job_labuser10764342_1756212917,851910675911544,651079287181128,2025-08-26T17:01:37.978Z,2025-08-26T17:02:37.681Z,customers_sales_summary,SUCCEEDED


Note: You can query, join, and filter different available system tables to find valuable insights. This is a broad topic and outside the scope of this course.

###Additional Resources
If you want to learn more about Jobs system tables, refer to the following documentation:
https://docs.databricks.com/aws/en/admin/system-tables/jobs#jobs

To explore the various available system tables, their relationships, and other related details, see:
https://docs.databricks.com/aws/en/admin/system-tables


&copy; 2026 Databricks, Inc. All rights reserved. Apache, Apache Spark, Spark, the Spark Logo, Apache Iceberg, Iceberg, and the Apache Iceberg logo are trademarks of the <a href="https://www.apache.org/" target="_blank">Apache Software Foundation</a>.<br/><br/><a href="https://databricks.com/privacy-policy" target="_blank">Privacy Policy</a> | <a href="https://databricks.com/terms-of-use" target="_blank">Terms of Use</a> | <a href="https://help.databricks.com/" target="_blank">Support</a>